# 가설 검증: 진열과 전단의 결합 효과

> 동일 상품·매장에서 진열과 전단을 함께 적용한 주차는 진열만 또는 전단만 적용한 가까운 주차보다 판매 발생률과 매출이 높을 것이다.

- H0: 진열+전단과 단독 노출의 판매 지표 차이가 0 이하이다.
- H1: 진열+전단의 판매 지표가 진열만 및 전단만보다 모두 높다.

## 고정 기준
- 분석 단위: 상품×매장×주차
- 처리: display≠0 및 mailer≠0
- 비교 1: display≠0, mailer=0
- 비교 2: display=0, mailer≠0
- 매칭: 동일 PRODUCT_ID·STORE_ID에서 가장 가까운 주차, 최대 ±4주
- 1차 결과: 판매 발생 여부
- 2차 결과: 매출·수량·장바구니 수

`causal_data.csv`에는 완전한 무노출 행이 없으므로 프로모션 대 무프로모션 효과는 검증하지 않습니다.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns",60)
PROJECT_DIR=Path.cwd().parent if Path.cwd().name=="notebooks" else Path.cwd()
OUTPUT_DIR=PROJECT_DIR/"data"/"processed"
TRANSACTION_PATH=PROJECT_DIR/"transaction_data.csv"
CAUSAL_PATH=PROJECT_DIR/"causal_data.csv"
PRODUCT_PATH=PROJECT_DIR/"product.csv"
CHUNK_SIZE=1_000_000
WEEK_TOLERANCE=4
RANDOM_SEED=42

## 1. 상품×매장×주차 판매 성과 생성

In [ ]:
tx=pd.read_csv(TRANSACTION_PATH,usecols=["BASKET_ID","PRODUCT_ID","STORE_ID","WEEK_NO","QUANTITY","SALES_VALUE"],
    dtype={"BASKET_ID":"int64","PRODUCT_ID":"int32","STORE_ID":"int32","WEEK_NO":"int16","QUANTITY":"int32","SALES_VALUE":"float32"})
key_cols=["PRODUCT_ID","STORE_ID","WEEK_NO"]
tx_week=tx.groupby(key_cols).agg(revenue=("SALES_VALUE","sum"),units=("QUANTITY","sum"),baskets=("BASKET_ID","nunique"))
print(f"판매가 발생한 상품×매장×주차: {len(tx_week):,}")

## 2. 같은 상품×매장의 가까운 주차 매칭

원본 노출 파일이 PRODUCT_ID 순으로 정렬돼 있으므로 마지막 상품을 다음 청크로 넘겨 상품 경계가 잘리지 않도록 처리합니다. 판매가 없던 노출 행은 성과를 0으로 유지합니다.

In [ ]:
OUTCOMES=["has_sale","revenue","units","baskets"]
def matched_effects(data,single_group,comparison_name):
    combo=data.loc[data["promo_group"].eq("진열+전단"),key_cols+OUTCOMES].copy()
    single=data.loc[data["promo_group"].eq(single_group),key_cols+OUTCOMES].copy()
    if combo.empty or single.empty:return pd.DataFrame()
    single["matched_week"]=single["WEEK_NO"]
    single=single.rename(columns={c:f"{c}_single" for c in OUTCOMES})
    combo=combo.sort_values("WEEK_NO"); single=single.sort_values("WEEK_NO")
    pairs=pd.merge_asof(combo,single,on="WEEK_NO",by=["PRODUCT_ID","STORE_ID"],direction="nearest",tolerance=WEEK_TOLERANCE)
    pairs=pairs.dropna(subset=["matched_week"])
    if pairs.empty:return pd.DataFrame()
    pairs["week_distance"]=(pairs["WEEK_NO"]-pairs["matched_week"]).abs()
    for c in OUTCOMES:pairs[f"{c}_diff"]=pairs[c]-pairs[f"{c}_single"]
    result=pairs.groupby(["PRODUCT_ID","STORE_ID"]).agg(
        matched_pairs=("WEEK_NO","size"),mean_week_distance=("week_distance","mean"),
        combo_sales_incidence=("has_sale","mean"),single_sales_incidence=("has_sale_single","mean"),
        combo_revenue=("revenue","mean"),single_revenue=("revenue_single","mean"),
        sales_incidence_diff=("has_sale_diff","mean"),revenue_diff=("revenue_diff","mean"),
        units_diff=("units_diff","mean"),baskets_diff=("baskets_diff","mean")
    ).reset_index()
    result["sales_incidence_relative_lift"]=result["combo_sales_incidence"]/result["single_sales_incidence"].replace(0,np.nan)
    result["revenue_relative_lift"]=result["combo_revenue"]/result["single_revenue"].replace(0,np.nan)
    result["comparison"]=comparison_name
    return result

def process_complete_products(causal):
    causal["display_flag"]=causal["display"].fillna("0").ne("0")
    causal["mailer_flag"]=causal["mailer"].fillna("0").ne("0")
    causal["promo_group"]=np.select([causal["display_flag"]&causal["mailer_flag"],causal["display_flag"],causal["mailer_flag"]],
        ["진열+전단","진열만","전단만"],default="확인 필요")
    causal=causal.join(tx_week,on=key_cols)
    for c in ["revenue","units","baskets"]:causal[c]=causal[c].fillna(0)
    causal["has_sale"]=(causal["revenue"]>0).astype(int)
    return [matched_effects(causal,"진열만","진열+전단 - 진열만"),matched_effects(causal,"전단만","진열+전단 - 전단만")]

In [ ]:
effect_parts=[]; carry=pd.DataFrame(); previous_last_product=None
dtype={"PRODUCT_ID":"int32","STORE_ID":"int32","WEEK_NO":"int16","display":"string","mailer":"string"}
for chunk_no,chunk in enumerate(pd.read_csv(CAUSAL_PATH,chunksize=CHUNK_SIZE,dtype=dtype),start=1):
    if previous_last_product is not None and chunk["PRODUCT_ID"].iloc[0]<previous_last_product:raise ValueError("causal_data가 PRODUCT_ID 순으로 정렬돼 있지 않습니다")
    previous_last_product=int(chunk["PRODUCT_ID"].iloc[-1])
    if not carry.empty:chunk=pd.concat([carry,chunk],ignore_index=True)
    last_product=chunk["PRODUCT_ID"].iloc[-1]
    carry=chunk.loc[chunk["PRODUCT_ID"].eq(last_product)].copy()
    complete=chunk.loc[~chunk["PRODUCT_ID"].eq(last_product)].copy()
    if not complete.empty:
        for part in process_complete_products(complete):
            if not part.empty:effect_parts.append(part)
    if chunk_no%10==0:print(f"{chunk_no}개 청크 처리 완료")
for part in process_complete_products(carry):
    if not part.empty:effect_parts.append(part)
promotion_product_store_effects=pd.concat(effect_parts,ignore_index=True)
print(f"매칭된 상품×매장 비교: {len(promotion_product_store_effects):,}")

## 3. 상품 단위 효과와 가설 검정

같은 상품이 여러 매장에 반복되므로 먼저 상품 단위 평균으로 집계한 뒤, 상품을 독립 단위로 근사한 단측 부호순열 검정을 수행합니다.

In [ ]:
product_effects=promotion_product_store_effects.groupby(["PRODUCT_ID","comparison"]).agg(
    product_stores=("STORE_ID","nunique"),matched_pairs=("matched_pairs","sum"),
    sales_incidence_diff=("sales_incidence_diff","mean"),revenue_diff=("revenue_diff","mean"),
    units_diff=("units_diff","mean"),baskets_diff=("baskets_diff","mean")
).reset_index()

def sign_flip_test(values,n_perm=20000):
    values=np.asarray(values,float); values=values[np.isfinite(values)]; n=len(values)
    if n<2:return {"products":n,"mean_effect":np.nan,"ci_low":np.nan,"ci_high":np.nan,"p_value_one_sided":np.nan}
    mean=values.mean(); se=values.std(ddof=1)/np.sqrt(n); rng=np.random.default_rng(RANDOM_SEED); extreme=0; done=0; batch=200
    while done<n_perm:
        size=min(batch,n_perm-done); signs=rng.choice([-1,1],size=(size,n)); extreme+=((signs*values).mean(axis=1)>=mean).sum(); done+=size
    return {"products":n,"mean_effect":mean,"ci_low":mean-1.96*se,"ci_high":mean+1.96*se,
            "p_value_one_sided":(1+extreme)/(n_perm+1)}
tests=[]
for comparison,group in product_effects.groupby("comparison"):
    for outcome,col in [("판매발생률","sales_incidence_diff"),("매출","revenue_diff"),("수량","units_diff"),("장바구니 수","baskets_diff")]:
        tests.append({"comparison":comparison,"outcome":outcome,**sign_flip_test(group[col])})
promotion_combination_hypothesis_tests=pd.DataFrame(tests)
display(promotion_combination_hypothesis_tests)
primary=promotion_combination_hypothesis_tests.query("outcome=='판매발생률'")
supported=(len(primary)==2 and (primary["mean_effect"]>0).all() and (primary["p_value_one_sided"]<.05).all())
print("판정: 가설을 지지합니다." if supported else "판정: 두 단독 방식과의 비교를 모두 통과하지 못해 가설을 지지할 충분한 근거가 없습니다.")

## 4. 카테고리별 결합 효과

In [ ]:
products=pd.read_csv(PRODUCT_PATH,usecols=["PRODUCT_ID","DEPARTMENT","COMMODITY_DESC"])
with_category=promotion_product_store_effects.merge(products,on="PRODUCT_ID",how="left",validate="many_to_one")
promotion_combination_category_effects=with_category.groupby(["DEPARTMENT","COMMODITY_DESC","comparison"],dropna=False).agg(
    product_stores=("STORE_ID","size"),products=("PRODUCT_ID","nunique"),matched_pairs=("matched_pairs","sum"),
    sales_incidence_diff=("sales_incidence_diff","mean"),revenue_diff=("revenue_diff","mean"),
    units_diff=("units_diff","mean"),baskets_diff=("baskets_diff","mean")
).reset_index()
reliable=promotion_combination_category_effects.query("product_stores>=30 and products>=5")
display(reliable.sort_values("revenue_diff",ascending=False).head(20))

## 5. 해석 주의사항

- 동일 상품·매장과 가까운 주차를 맞춰 상품·매장 차이와 큰 시간 차이를 줄였지만, 프로모션 배정은 무작위가 아닙니다.
- 진열+전단 주차가 특별 행사 기간과 겹치는 등 관측되지 않은 차이가 남을 수 있습니다.
- 통계적으로 유의해도 프로모션 비용이 없으므로 수익성 우월까지 확정할 수 없습니다.
- 실제 적용 전 매장·상품을 무작위 배정한 A/B 테스트가 필요합니다.

In [ ]:
promotion_product_store_effects.to_csv(OUTPUT_DIR/"promotion_matched_product_store_effects.csv",index=False,encoding="utf-8-sig")
promotion_combination_hypothesis_tests.to_csv(OUTPUT_DIR/"promotion_combination_hypothesis_tests.csv",index=False,encoding="utf-8-sig")
promotion_combination_category_effects.to_csv(OUTPUT_DIR/"promotion_combination_category_effects.csv",index=False,encoding="utf-8-sig")
print("가설 검증 결과 3개 저장 완료")